In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
while not (PROJECT_ROOT / 'setup.py').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Cannot find project root')
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from solve_dae.integrate import solve_dae

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "scripts" / "Experimental"))

from simulate_HFR import build_cell, build_atol, soc_cutoff_event
from rfbmodel1D.parameters.params import cathodeparams, anodeparams, membraneparams, cellparams

## Test selection

Set `TEST_NO` to the test number whose fitted HFR (membrane) and, if available, E-cell
(cathode/anode kinetics) parameters should be loaded. `results/test fitting/global/` currently
only has an E-cell fit covering test 16 — for any other test the notebook falls back to the
`cathodeparams`/`anodeparams` defaults and prints a warning.

In [ ]:
# -- Test to inspect ---------------------------------------------------------
TEST_NO = 16

# -- Solver settings ----------------------------------------------------------
TF     = 1800   # [s] max simulated time per operating point
RTOL   = 1e-5
STRIDE = 1       # subsample experimental current points (1 = use all)
N_FILL = 0       # extra interior current points per gap, for a smoother HFR curve (0 = off)

## Load this test's HFR + E-cell fits

In [ ]:
def _resolve_ecell_params(test_no: int):
    """k_ca (cathode reaction_rate_constant) and j0_an (anode j_0) for this
    test, resolved in the same priority order as
    scripts/Experimental/summarize_ecell_fits.py:
      1) a per-test Stage-2 fit saved in the test's own folder
      2) the most recent global Stage-1 fit that covered this test
    Returns (k_ca, j0_an, source_filename) or (None, None, None) if no
    E-cell fit is available for this test yet.
    """
    test_dir = ROOT / "results" / "test fitting" / f"test {test_no}"

    stage2 = sorted(test_dir.glob(f"global_j0an_test{test_no}_*.json"))
    if stage2:
        with open(stage2[-1]) as f:
            fit = json.load(f)
        return float(fit["params"]["k_ca"]), float(fit["params"]["j0_an"]), stage2[-1].name

    global_dir  = ROOT / "results" / "test fitting" / "global"
    global_fits = sorted(global_dir.glob("global_k_ca_*.json"))
    for gp in reversed(global_fits):   # most recent first
        with open(gp) as f:
            fit = json.load(f)
        j0_dict = fit["params"].get("j0_an", {})
        if str(test_no) in j0_dict:
            return float(fit["params"]["k_ca"]), float(j0_dict[str(test_no)]), gp.name

    return None, None, None


# -- HFR (membrane) fit --------------------------------------------------------
fit_dir  = ROOT / "results" / "test fitting" / f"test {TEST_NO}"
fit_path = sorted(fit_dir.glob(f"fit_v2_test{TEST_NO}_*.json"))[-1]
with open(fit_path) as f:
    _hfr = json.load(f)
_p, _s = _hfr["params"], _hfr["settings"]

hfr_params = dict(
    k_sorp_vw   = _p["k_vw_sorp"],
    k_desorp_vw = _p["S_desorp"] * _p["k_vw_sorp"],
    k_sorp_lw   = _p["k_lw_sorp"],
    k_desorp_lw = _p["S_desorp"] * _p["k_lw_sorp"],
    k_ca_lw     = _p["k_ca"],
    sf_D        = _s["sf_D"],
    vanadium_uptake = _s.get("vanadium_uptake", False),
)
R_EXTRA = _p["R_extra (\u03a9\u00b7cm\u00b2)"] / 1e4   # \u03a9\u00b7cm\u00b2 -> \u03a9\u00b7m\u00b2

# -- E-cell (cathode/anode kinetics) fit, if one exists for this test ---------
k_ca, j0_an, ecell_source = _resolve_ecell_params(TEST_NO)
if k_ca is None:
    print(f"[WARNING] No E-cell fit found for test {TEST_NO} \u2014 "
          f"using cathodeparams/anodeparams defaults.")
    k_ca  = cathodeparams.reaction_rate_constant
    j0_an = anodeparams.j_0

# -- Experimental operating point + current sweep ------------------------------
csv = ROOT / "data" / "experimental" / "HFR.csv"
_df = pd.read_csv(csv)
d   = _df[_df["test_no"] == TEST_NO]
soc = float(d["SOC"].iloc[0])
T   = float(d["T(\u00b0C)"].iloc[0])
Q_v = float(d["Q_v (ml/min)"].iloc[0])

j_charge    = d[d["mode"] == "charge"]   ["j_appl (A/m2)"].values[::STRIDE]
j_discharge = d[d["mode"] == "discharge"]["j_appl (A/m2)"].values[::STRIDE]
HFR_exp_ch  = d[d["mode"] == "charge"]   ["value (ohm cm2)"].values[::STRIDE]
HFR_exp_dis = d[d["mode"] == "discharge"]["value (ohm cm2)"].values[::STRIDE]

cfg = dict(
    soc=soc, T=T, Q_v=Q_v,
    j_charge=j_charge, j_discharge=j_discharge,
    HFR_exp_ch=HFR_exp_ch, HFR_exp_dis=HFR_exp_dis,
    hfr=hfr_params, R_EXTRA=R_EXTRA,
    k_ca=k_ca, j0_an=j0_an,
)

print(f"Test {TEST_NO}: SOC={soc:.0%}  T={T}\u00b0C  Q_v={Q_v} ml/min")
print(f"  HFR fit    -> {fit_path.name}")
print(f"  E-cell fit -> {ecell_source or '(defaults)'}")
print(f"  k_ca={k_ca:.3e}  j0_an={j0_an:.3e}  R_extra={R_EXTRA*1e4:.4f} \u03a9\u00b7cm\u00b2")
print(f"  {len(j_charge)} charge / {len(j_discharge)} discharge experimental points")

## Run the HFR simulation across the experimental current sweep

Each steady-state DAE solve already yields the converged membrane water-content (λ) profile
as part of computing HFR, so the ionomer behaviour below reuses these same runs instead of a
separate resistance calculation.

In [ ]:
def _apply_params(cfg: dict) -> None:
    """Apply this test's membrane, R_extra and kinetic params to the module-level
    singleton dataclasses that build_cell() reads from."""
    h = cfg["hfr"]
    membraneparams.k_sorp_vw   = h["k_sorp_vw"]
    membraneparams.k_desorp_vw = h["k_desorp_vw"]
    membraneparams.k_sorp_lw   = h["k_sorp_lw"]
    membraneparams.k_desorp_lw = h["k_desorp_lw"]
    membraneparams.k_ca_lw     = h["k_ca_lw"]
    membraneparams.sf_D        = h["sf_D"]
    cellparams.vanadium_uptake = h["vanadium_uptake"]
    cathodeparams.reaction_rate_constant = cfg["k_ca"]
    anodeparams.j_0                      = cfg["j0_an"]


def run_sim(j_appl: float, cfg: dict):
    """Run one steady-state operating point. Returns (HFR [\u03a9\u00b7cm\u00b2], lam_profile, x_mem)."""
    _apply_params(cfg)
    cell, y0, yp0 = build_cell(cfg["soc"], j_appl, cfg["T"], cfg["Q_v"])
    atol          = build_atol(cell)
    jac_sparsity  = cell.compute_jac_sparsity(y0, yp0)
    t0 = time.time()
    sol = solve_dae(
        cell.dae_residual_full_cell,
        (0.0, TF), y0, yp0,
        method="BDF", atol=atol, rtol=RTOL,
        events=soc_cutoff_event, jac_sparsity=jac_sparsity,
    )
    cell.dae_residual_full_cell(sol.t[-1], sol.y[:, -1], sol.yp[:, -1])
    if sol.t[-1] < 0.99 * TF:
        print(f"  Warning: t_final={sol.t[-1]:.1f} s (event triggered)")

    state       = cell.split_state(sol.y[:, -1])
    lam_profile = state["membrane"][:, 0]
    x_mem       = cell.mesh.membrane_mesh.x_grid

    kappa_io_ca = cell.cathode.kappa_io
    sigma_ca    = np.ones_like(kappa_io_ca) * cell.cathode.eff_conductivity
    dx_ca       = cell.mesh.cathode_mesh.dx * cell.cathode.thickness
    R_cathode   = sum(dx_ca / (sigma_ca + kappa_io_ca))

    kappa_io_m = cell.membrane.kappa_io
    dx_m       = cell.mesh.membrane_mesh.dx * cell.membrane.thickness
    R_membrane = sum(dx_m / kappa_io_m)

    R_ACL  = cell.anode.thickness  / cell.anode.eff_conductivity
    R_AGDL = cell.agdl.thickness   / cell.agdl.eff_conductivity
    R_BP   = cell.thickness_BP / cell.sigma_BP
    R_CC   = cell.thickness_CC / cell.sigma_CC
    HFR    = (R_cathode + R_membrane + R_ACL + R_AGDL + R_BP + R_CC + cfg["R_EXTRA"]) * 1e4

    print(f"  j={j_appl:+8.0f} A/m\u00b2  HFR={HFR:.4f} \u03a9\u00b7cm\u00b2  ({time.time()-t0:.1f}s)")
    return HFR, lam_profile, x_mem


def interior_points(arr, n=1):
    """n equally-spaced interior points between each consecutive pair in arr."""
    if n == 0 or len(arr) < 2:
        return np.array([])
    pts = []
    for a, b in zip(arr[:-1], arr[1:]):
        pts.extend(np.linspace(a, b, n + 2)[1:-1])
    return np.array(pts)

In [ ]:
print(f"Test {TEST_NO}  |  k_ca={cfg['k_ca']:.3e}  j0_an={cfg['j0_an']:.3e}")

print("-- Charge --")
_res_ch = [run_sim(float(j), cfg) for j in cfg["j_charge"]]
HFR_ch  = [r[0] for r in _res_ch]
lam_ch  = [r[1] for r in _res_ch]
x_mem   = _res_ch[0][2]

j_ch_fill   = interior_points(cfg["j_charge"], n=N_FILL)
HFR_ch_fill = [run_sim(float(j), cfg)[0] for j in j_ch_fill]
j_model_ch   = np.concatenate([cfg["j_charge"], j_ch_fill])
HFR_model_ch = np.concatenate([HFR_ch, HFR_ch_fill])
idx = np.argsort(j_model_ch)
j_model_ch, HFR_model_ch = j_model_ch[idx], HFR_model_ch[idx]

print("-- Discharge --")
_res_dis = [run_sim(-float(j), cfg) for j in cfg["j_discharge"]]
HFR_dis  = [r[0] for r in _res_dis]
lam_dis  = [r[1] for r in _res_dis]

j_dis_fill   = interior_points(cfg["j_discharge"], n=N_FILL)
HFR_dis_fill = [run_sim(-float(j), cfg)[0] for j in j_dis_fill]
j_model_dis   = np.concatenate([cfg["j_discharge"], j_dis_fill])
HFR_model_dis = np.concatenate([HFR_dis, HFR_dis_fill])
idx = np.argsort(j_model_dis)
j_model_dis, HFR_model_dis = j_model_dis[idx], HFR_model_dis[idx]

HFR_exp_all = np.concatenate([cfg["HFR_exp_ch"], cfg["HFR_exp_dis"]])
HFR_sim_all = np.array(HFR_ch + HFR_dis)
mse = float(np.mean((HFR_sim_all - HFR_exp_all) ** 2))
ape = float(np.mean(np.abs((HFR_sim_all - HFR_exp_all) / HFR_exp_all)) * 100)
print(f"\nMSE={mse:.6f}  RMSE={mse**0.5:.4f} \u03a9\u00b7cm\u00b2  APE={ape:.2f}%")

sim_results = dict(
    j_model_ch=j_model_ch, HFR_model_ch=HFR_model_ch,
    j_model_dis=j_model_dis, HFR_model_dis=HFR_model_dis,
    HFR_sim_ch=HFR_ch, HFR_sim_dis=HFR_dis,
    lam_ch=lam_ch, lam_dis=lam_dis, x_mem=x_mem,
    mse=mse, ape=ape,
)

## HFR fit-quality check

In [ ]:
plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',
    'font.size':         12,
    'axes.labelsize':    13,
    'axes.titlesize':    12,
    'xtick.labelsize':   11,
    'ytick.labelsize':   11,
    'legend.fontsize':   10,
    'axes.linewidth':    0.8,
    'xtick.direction':   'in',
    'ytick.direction':   'in',
    'xtick.top':         True,
    'ytick.right':       True,
    'lines.linewidth':   1.4,
})

fig, ax = plt.subplots(figsize=(5, 4.2))

ax.plot(cfg["j_discharge"], cfg["HFR_exp_dis"],
        'o', ms=6, mfc='none', mec='k', mew=1.0, label='Exp., dis.')
ax.plot(cfg["j_charge"],    cfg["HFR_exp_ch"],
        'o', ms=6, mfc='k',    mec='k', mew=1.0, label='Exp., ch.')
ax.plot(sim_results["j_model_ch"],  sim_results["HFR_model_ch"],  '-', color='C0', label='Model, ch.')
ax.plot(sim_results["j_model_dis"], sim_results["HFR_model_dis"], '-', color='C3', label='Model, dis.')

ax.set_xlabel(r'Current density / A m$^{-2}$')
ax.set_ylabel(r'HFR / $\Omega$ cm$^2$')
ax.set_title(
    f"Test {TEST_NO}  |  SOC {cfg['soc']:.0%}  T={cfg['T']:.0f}\u00b0C  Q$_v$={cfg['Q_v']:.0f} ml/min\n"
    f"RMSE = {sim_results['mse']**0.5:.4f} $\Omega$ cm$^2$  |  APE = {sim_results['ape']:.2f}%"
)
ax.legend(frameon=False)
ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

fig.tight_layout()
out_path = ROOT / "figures" / f"ionomer_test{TEST_NO}_HFR.pdf"
out_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved -> {out_path}")

## Membrane water-content (λ) profile

λ(x) across the membrane thickness at the final (steady) simulated state for every
experimental current density, x=0 at the cathode interface and x=1 at the anode interface.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)

nc, nd = len(cfg["j_charge"]), len(cfg["j_discharge"])
colors_c = plt.cm.Blues(np.linspace(0.4, 0.95, max(nc, 1)))
colors_d = plt.cm.Reds(np.linspace(0.4, 0.95, max(nd, 1)))

for lam, j, c in zip(sim_results["lam_ch"], cfg["j_charge"], colors_c):
    axes[0].plot(sim_results["x_mem"], lam, color=c, label=f'{j:.0f} A/m\u00b2')
axes[0].set_xlabel('Normalised membrane position (cathode \u2192 anode)')
axes[0].set_ylabel(r'Water content $\lambda$')
axes[0].set_title('Charge')
axes[0].legend(fontsize=8, loc='best')
axes[0].grid(True, alpha=0.3)

for lam, j, c in zip(sim_results["lam_dis"], cfg["j_discharge"], colors_d):
    axes[1].plot(sim_results["x_mem"], lam, color=c, label=f'{j:.0f} A/m\u00b2')
axes[1].set_xlabel('Normalised membrane position (cathode \u2192 anode)')
axes[1].set_title('Discharge')
axes[1].legend(fontsize=8, loc='best')
axes[1].grid(True, alpha=0.3)

fig.suptitle(
    f"Test {TEST_NO}  |  SOC {cfg['soc']:.0%}  T={cfg['T']:.0f}\u00b0C  Q$_v$={cfg['Q_v']:.0f} ml/min",
    fontsize=12,
)
fig.tight_layout()
out_path = ROOT / "figures" / f"ionomer_test{TEST_NO}_lambda.pdf"
fig.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved -> {out_path}")